In [2]:
from collections import Counter
from pathlib import Path

import onnx
from onnx import TensorProto

BASE = Path("onnx_streaming_vocos")
MODEL_PATHS = sorted(BASE.glob("*.onnx"))


def dtype_name(elem_type: int) -> str:
    return TensorProto.DataType.Name(elem_type)


def shape_of(value_info) -> list[str]:
    dims = []
    for dim in value_info.type.tensor_type.shape.dim:
        if dim.dim_param:
            dims.append(dim.dim_param)
        elif dim.dim_value:
            dims.append(str(dim.dim_value))
        else:
            dims.append("?")
    return dims


def print_model_summary(model_path: Path) -> None:
    model = onnx.load(model_path)
    graph = model.graph
    op_counts = Counter(node.op_type for node in graph.node)

    print(f"FILE {model_path}")
    print(f"  ir_version: {model.ir_version}")
    print(
        "  opsets:",
        ", ".join(f"{op.domain or 'ai.onnx'}:{op.version}" for op in model.opset_import),
    )
    print("  inputs:")
    for value_info in graph.input:
        print(
            f"    - {value_info.name}: dtype={dtype_name(value_info.type.tensor_type.elem_type)}, "
            f"shape={shape_of(value_info)}"
        )
    print("  outputs:")
    for value_info in graph.output:
        print(
            f"    - {value_info.name}: dtype={dtype_name(value_info.type.tensor_type.elem_type)}, "
            f"shape={shape_of(value_info)}"
        )
    print(f"  initializers: {len(graph.initializer)}")
    print(f"  nodes: {len(graph.node)}")
    print(
        "  op_counts:",
        ", ".join(f"{op_type}={count}" for op_type, count in sorted(op_counts.items())),
    )

    gelu_nodes = []
    for index, node in enumerate(graph.node):
        node_name = node.name or ""
        if node.op_type.lower() == "gelu" or "gelu" in node_name.lower():
            gelu_nodes.append(
                {
                    "index": index,
                    "op_type": node.op_type,
                    "name": node.name or "<unnamed>",
                    "inputs": list(node.input),
                    "outputs": list(node.output),
                }
            )

    if gelu_nodes:
        print("  gelu_nodes:")
        for node in gelu_nodes:
            print(
                f"    - idx={node['index']}, op_type={node['op_type']}, name={node['name']}"
            )
            print(f"      inputs={node['inputs']}")
            print(f"      outputs={node['outputs']}")
    else:
        print("  gelu_nodes: none")

    print()


if not MODEL_PATHS:
    raise FileNotFoundError(f"No *_fp16.onnx files found under {BASE}")

for model_path in MODEL_PATHS:
    print_model_summary(model_path)


FILE onnx_streaming_vocos/acoustic_expand.onnx
  ir_version: 9
  opsets: ai.onnx:19
  inputs:
    - d_enc_expanded: dtype=FLOAT, shape=['1', '640', '543']
  outputs:
    - en: dtype=FLOAT, shape=['1', '543', '512']
  initializers: 3
  nodes: 25
  op_counts: Concat=2, Constant=10, Expand=2, Gather=2, LSTM=1, Reshape=1, Shape=2, Transpose=3, Unsqueeze=2
  gelu_nodes: none

FILE onnx_streaming_vocos/acoustic_expand_fp16.onnx
  ir_version: 9
  opsets: ai.onnx:19
  inputs:
    - d_enc_expanded: dtype=FLOAT16, shape=['1', '640', '543']
  outputs:
    - en: dtype=FLOAT16, shape=['1', '543', '512']
  initializers: 5
  nodes: 5
  op_counts: LSTM=1, Reshape=1, Transpose=3
  gelu_nodes: none

FILE onnx_streaming_vocos/acoustic_expand_opt.onnx
  ir_version: 9
  opsets: ai.onnx:19
  inputs:
    - d_enc_expanded: dtype=FLOAT, shape=['1', '640', '543']
  outputs:
    - en: dtype=FLOAT, shape=['1', '543', '512']
  initializers: 5
  nodes: 5
  op_counts: LSTM=1, Reshape=1, Transpose=3
  gelu_nodes: non